In [1]:
import pandas as pd
from scipy import stats
import numpy as np
from functools import partial

from data_preprocessing import HeartbeatDataProcessor 

import sys
sys.path.insert(1, '../analysis_and_validation')

from read_data import load_subject

In [2]:
df = load_subject(101)

Total structured columns generated: 54
First 10 columns: ['timestamp', 'activity_id', 'heart_rate', 'hand_temp', 'hand_acc16_x', 'hand_acc16_y', 'hand_acc16_z', 'hand_acc6_x', 'hand_acc6_y', 'hand_acc6_z']
Reading from ../data/PAMAP2_Dataset/Protocol/subject101.dat
Loading raw subject file (this might take a few seconds because it is a massive telemetry file)...

--- RAW DATA DIAGNOSTICS FOR SUBJECT 101 ---
Total raw rows: 376,417
Total continuous tracking time: 62.88 minutes

Top 5 Columns with the Highest Percentage of Missing Data (NaNs):
heart_rate      90.864121
hand_temp        0.386274
hand_acc16_x     0.386274
hand_acc16_y     0.386274
hand_acc6_x      0.386274
dtype: float64


In [3]:
def _peak2peak_amp(window_df):
    return window_df.agg(max) - window_df.agg(min)
def _sum_of_area(window_df):
    return window_df.agg(abs).agg(np.sum)
def _signal_mean_energy(window_df):
    #may be a neater way to do this
    df_sq = window_df**2
    return df_sq.agg('mean')

In [4]:
features = df.agg(func=['mean',
                        stats.hmean,
                        'std',
                        'max',
                        'min',
                        _peak2peak_amp,
                        'median',
                        partial(stats.median_abs_deviation,nan_policy='omit'),
                        partial(stats.iqr,nan_policy='omit'),
                        _sum_of_area,
                        _signal_mean_energy,
                        'skew',
                        'kurtosis'])

In [5]:
features

,timestamp,activity_id,heart_rate,hand_temp,hand_acc16_x,hand_acc16_y,hand_acc16_z,hand_acc6_x,hand_acc6_y,hand_acc6_z,...,ankle_gyro_x,ankle_gyro_y,ankle_gyro_z,ankle_mag_x,ankle_mag_y,ankle_mag_z,ankle_orient_w,ankle_orient_x,ankle_orient_y,ankle_orient_z
mean,1.890460e+03,5.525765e+00,1.241355e+02,3.242990e+01,-3.349118e+00,6.278528e+00,3.407735e+00,-3.290532e+00,6.303530e+00,3.572275e+00,...,-0.000110,0.000361,0.013872,-5.087307e+01,-4.103087e+00,1.043377e+01,1.0,0.0,0.0,0.0
hmean,6.160415e+02,0.000000e+00,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
std,1.086624e+03,6.617002e+00,2.400034e+01,1.236379e+00,6.365465e+00,5.196436e+00,3.383241e+00,6.395511e+00,5.217042e+00,3.369418e+00,...,0.996037,0.603546,1.786058,2.572473e+01,3.061531e+01,2.952419e+01,0.0,0.0,0.0,0.0
max,3.772540e+03,2.400000e+01,1.830000e+02,3.387500e+01,2.614580e+01,1.068980e+02,1.322950e+02,1.911070e+01,6.206060e+01,5.361710e+01,...,13.142500,6.410380,9.377020,9.752550e+01,1.233060e+02,1.248060e+02,1.0,0.0,0.0,0.0
min,8.380000e+00,0.000000e+00,7.800000e+01,3.000000e+01,-1.276490e+02,-8.123440e+01,-3.412480e+01,-5.698420e+01,-4.279720e+01,-2.205220e+01,...,-11.688700,-7.807450,-11.619400,-1.728650e+02,-1.379080e+02,-1.092890e+02,1.0,0.0,0.0,0.0
_peak2peak_amp,3.764160e+03,2.400000e+01,1.050000e+02,3.875000e+00,1.537948e+02,1.881324e+02,1.664198e+02,7.609490e+01,1.048578e+02,7.566930e+01,...,24.831200,14.217830,20.996420,2.703905e+02,2.612140e+02,2.340950e+02,0.0,0.0,0.0,0.0
median,1.890460e+03,3.000000e+00,1.250000e+02,3.281250e+01,-2.982460e+00,6.009140e+00,3.464280e+00,-2.923560e+00,6.049510e+00,3.674220e+00,...,-0.001233,0.003658,-0.002747,-5.324985e+01,-5.071440e+00,1.109790e+01,1.0,0.0,0.0,0.0
median_abs_deviation,9.410400e+02,3.000000e+00,1.500000e+01,8.125000e-01,4.751980e+00,2.522400e+00,2.017610e+00,4.752690e+00,2.490900e+00,2.010870e+00,...,0.113761,0.090394,0.140151,1.375350e+01,2.442756e+01,2.148115e+01,0.0,0.0,0.0,0.0
iqr,1.882080e+03,7.000000e+00,3.200000e+01,2.062500e+00,9.509300e+00,5.029890e+00,4.049830e+00,9.487965e+00,4.969830e+00,4.033220e+00,...,0.258793,0.215972,0.365552,2.714457e+01,4.921975e+01,4.301456e+01,0.0,0.0,0.0,0.0
_sum_of_area,7.116013e+08,2.079992e+06,4.268895e+06,1.216001e+07,2.074340e+06,2.463940e+06,1.494053e+06,2.074131e+06,2.474980e+06,1.545116e+06,...,194290.102400,124855.423605,347666.226474,1.926698e+07,9.637049e+06,9.529324e+06,375090.0,0.0,0.0,0.0
